In [ ]:
!pip install -q resemblyzer soundfile transformers librosa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 96.9 MB/s eta 0:00:00


In [ ]:
import os

# 1. Setup Kaggle API Credentials
os.environ['KAGGLE_USERNAME'] = "notalijafar" # <--- Replace this
os.environ['KAGGLE_KEY'] = ""           # <--- Replace this

# 2. Install required libraries
!pip install -q kaggle resemblyzer soundfile transformers librosa accelerate

# 3. Download and unzip the dataset directly into Colab
print("Downloading dataset from Kaggle...")
!kaggle datasets download -d muhammadahmedansari/urdu-dataset-20000 --unzip -p /content/dataset

print("Download and installation complete!")

Dataset URL: https://www.kaggle.com/datasets/muhammadahmedansari/urdu-dataset-20000
License(s): other
100% 3.95G/3.95G [00:47<00:00, 88.5MB/s]

Download and installation complete!


In [ ]:
!find /content/dataset -maxdepth 3

Streaming output truncated to the last 5000 lines.
/content/dataset/limited_wav_files/limited_wav_files/common_voice_ur_31969598.wav
/content/dataset/limited_wav_files/limited_wav_files/common_voice_ur_31969603.wav
/content/dataset/limited_wav_files/limited_wav_files/common_voice_ur_31969605.wav
/content/dataset/limited_wav_files/limited_wav_files/common_voice_ur_31969606.wav
/content/dataset/limited_wav_files/limited_wav_files/common_voice_ur_31969608.wav
/content/dataset/limited_wav_files/limited_wav_files/common_voice_ur_31969613.wav
/content/dataset/limited_wav_files/limited_wav_files/common_voice_ur_31969617.wav
/content/dataset/limited_wav_files/limited_wav_files/common_voice_ur_31969631.wav
/content/dataset/limited_wav_files/limited_wav_files/common_voice_ur_31969642.wav
/content/dataset/limited_wav_files/limited_wav_files/common_voice_ur_31969646.wav
/content/dataset/limited_wav_files/limited_wav_files/common_voice_ur_31969648.wav
/content/dataset/limited_wav_files/limited_wav_

In [ ]:
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import os
import glob
import librosa
from pathlib import Path
from scipy.signal import resample
from transformers import VitsModel, AutoTokenizer
from resemblyzer import VoiceEncoder, preprocess_wav
from google.colab import userdata
from huggingface_hub import login

# --- 1. SETTINGS & AUTHENTICATION ---
NUM_SAMPLES = 30
MODEL_NAME = "facebook/mms-tts-urd-script_arabic"
OUTPUT_DIR = "/content/mushra_urdu20k"

# Login to Hugging Face using Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
except Exception as e:
    print("HF_TOKEN secret not found or login failed. Proceeding without it.")
    hf_token = None

# --- 2. LOCATE DATA ---
print("\nLocating dataset files...")
TSV_PATH = "/content/dataset/final_main_dataset.tsv"
# Updated based on the 'find' results
AUDIO_FOLDER = "/content/dataset/limited_wav_files/limited_wav_files"

if not os.path.exists(TSV_PATH):
    raise FileNotFoundError(f"Could not find the TSV file at {TSV_PATH}")

df = pd.read_csv(TSV_PATH, sep='\t')
text_col = "sentence" if "sentence" in df.columns else "text"
audio_col = "path" if "path" in df.columns else "client_id"

# --- 3. LOAD MODELS (GPU) ---
print("\nLoading models to GPU...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
model = VitsModel.from_pretrained(MODEL_NAME, token=hf_token).to(device)
model.eval()
encoder = VoiceEncoder()

# --- 4. PREPARE DIRECTORIES ---
for sub in ["reference", "generated", "anchor"]:
    os.makedirs(f"{OUTPUT_DIR}/{sub}", exist_ok=True)

results = []
eval_df = df.head(NUM_SAMPLES)

# --- 5. GENERATION & EVALUATION ---
print(f"\nStarting generation for {NUM_SAMPLES} samples...")

for i, row in eval_df.iterrows():
    try:
        text = row[text_col]
        filename = str(row[audio_col])

        # Correct the extension: the TSV lists .mp3, but the files are actually .wav
        if filename.endswith('.mp3'):
            filename = filename.replace('.mp3', '.wav')
        elif not filename.endswith('.wav'):
            filename += '.wav'

        original_path = os.path.join(AUDIO_FOLDER, filename)

        if not os.path.exists(original_path):
            print(f"Warning: File not found {original_path}")
            continue

        # 1. Reference Audio
        ref_array, ref_sr = librosa.load(original_path, sr=None)
        ref_path = f"{OUTPUT_DIR}/reference/sample_{i}.wav"
        sf.write(ref_path, ref_array, ref_sr)

        # 2. Generated Audio
        inputs = tokenizer(text, return_tensors="pt").to(device)
        with torch.no_grad():
            output = model(**inputs)
        gen_speech = output.waveform[0].cpu().numpy()
        gen_sr = model.config.sampling_rate
        gen_path = f"{OUTPUT_DIR}/generated/sample_{i}.wav"
        sf.write(gen_path, gen_speech, gen_sr)

        # 3. Anchor Audio
        target_len_8k = int(len(ref_array) * 8000 / ref_sr)
        downsampled = resample(ref_array, target_len_8k)
        upsampled = resample(downsampled, len(ref_array))
        anc_path = f"{OUTPUT_DIR}/anchor/sample_{i}.wav"
        sf.write(anc_path, upsampled, ref_sr)

        # 4. Metrics
        wav_ref = preprocess_wav(Path(ref_path))
        wav_gen = preprocess_wav(Path(gen_path))

        mid = len(wav_ref) // 2
        emb_ref_full = encoder.embed_utterance(wav_ref)
        emb_ref_a = encoder.embed_utterance(wav_ref[:mid])
        emb_ref_b = encoder.embed_utterance(wav_ref[mid:])
        emb_gen = encoder.embed_utterance(wav_gen)

        sim = np.dot(emb_ref_full, emb_gen) / (np.linalg.norm(emb_ref_full) * np.linalg.norm(emb_gen))
        self_sim = np.dot(emb_ref_a, emb_ref_b) / (np.linalg.norm(emb_ref_a) * np.linalg.norm(emb_ref_b))

        results.append({
            "sample_id": i,
            "urdu_text": text,
            "ref_gen_similarity": float(sim),
            "ref_self_similarity": float(self_sim)
        })
        print(f"Sample {i+1}/{NUM_SAMPLES} complete. Similarity: {sim:.4f}")

    except Exception as e:
        print(f"Skipping sample {i} due to error: {e}")
        continue

# --- 6. SAVE RESULTS ---
results_df = pd.DataFrame(results)
csv_path = "/content/urdu20k_mms_evaluation_stats.csv"
results_df.to_csv(csv_path, index=False)

print("\n=== EVALUATION SUMMARY ===")
if not results_df.empty:
    print(f"Mean Similarity (Reference vs Generated): {results_df['ref_gen_similarity'].mean():.4f}")
    print(f"Mean Self-Similarity (Reference A vs B): {results_df['ref_self_similarity'].mean():.4f}")
else:
    print("No samples were processed successfully.")


Locating dataset files...

Loading models to GPU...
Active Device: cuda


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Loaded the voice encoder model on cuda in 0.02 seconds.

Starting generation for 30 samples...
Sample 1/30 complete. Similarity: 0.5116
Sample 2/30 complete. Similarity: 0.5948
Sample 3/30 complete. Similarity: 0.5615
Sample 4/30 complete. Similarity: 0.4704
Sample 5/30 complete. Similarity: 0.6023
Sample 6/30 complete. Similarity: 0.5980
Sample 7/30 complete. Similarity: 0.6479
Sample 8/30 complete. Similarity: 0.6329
Sample 9/30 complete. Similarity: 0.6898
Sample 10/30 complete. Similarity: 0.5822
Sample 11/30 complete. Similarity: 0.5043
Sample 12/30 complete. Similarity: 0.6795
Sample 13/30 complete. Similarity: 0.5637
Sample 14/30 complete. Similarity: 0.6102
Sample 15/30 complete. Similarity: 0.5388
Sample 16/30 complete. Similarity: 0.6454
Sample 17/30 complete. Similarity: 0.4704
Sample 18/30 complete. Similarity: 0.6393
Sample 19/30 complete. Similarity: 0.6103
Sample 20/30 complete. Similarity: 0.5899
Sample 21/30 complete. Similarity: 0.6001
Sample 22/30 complete. Similarit

In [ ]:
import shutil
from google.colab import files

zip_filename = "/content/mushra_urdu20k_samples"

# 1. Zip the audio folders (Reference, Generated, Anchor)
print(f"Creating archive of samples from {OUTPUT_DIR}...")
shutil.make_archive(zip_filename, 'zip', OUTPUT_DIR)

# 2. Trigger automatic downloads
print("Downloading files to your computer...")
files.download(f"{zip_filename}.zip")
files.download("/content/urdu20k_mms_evaluation_stats.csv")

print("Download process started!")

Creating archive of samples from /content/mushra_urdu20k...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download process started!
